# Pertemuan 14 — Teori Kompleksitas, SAT, dan Inferensi Logika
Reduksi polinomial, solver DPLL, *phase transition* 3-SAT, dan **[Modern]** inferensi klausa Horn (forward chaining linear, inti sistem pakar).

In [ ]:
import itertools, random, math, time
from collections import defaultdict, deque
import numpy as np
import matplotlib.pyplot as plt
random.seed(14)

## 1. SAT: brute force (2ⁿ) dan DPLL
Klausa = list integer (positif = variabel, negatif = negasi), mis. `[1, -2, 3]` = (x₁ ∨ ¬x₂ ∨ x₃).

In [ ]:
def brute_sat(clauses, n):
    for bits in itertools.product([False, True], repeat=n):
        if all(any(bits[abs(l)-1] == (l > 0) for l in c) for c in clauses): return True
    return False

def dpll(clauses, stats, assign=None):
    assign = dict(assign or {})
    while True:                                                  # unit propagation
        unit = None; new = []
        for c in clauses:
            if any(assign.get(abs(l)) == (l > 0) for l in c): continue        # klausa terpenuhi
            rest = [l for l in c if abs(l) not in assign]
            if not rest: return False                                          # konflik
            if len(rest) == 1 and unit is None: unit = rest[0]
            new.append(rest)
        clauses = new
        if unit is None: break
        assign[abs(unit)] = unit > 0
    if not clauses: return True
    lit = clauses[0][0]; stats["decisions"] += 1
    return dpll(clauses, stats, {**assign, abs(lit): lit > 0}) or dpll(clauses, stats, {**assign, abs(lit): lit < 0})

def random_3sat(n, m):
    return [[v * random.choice([-1, 1]) for v in random.sample(range(1, n + 1), 3)] for _ in range(m)]

for _ in range(200):                                            # verifikasi vs brute force
    n = 8; cl = random_3sat(n, random.randint(10, 45)); assert dpll(cl, {"decisions": 0}) == brute_sat(cl, n)
print("DPLL cocok dengan brute force pada 200 instans ✔")

## 2. Phase transition 3-SAT
Probabilitas *satisfiable* turun tajam sekitar rasio klausa/variabel ≈ 4.26; instans di sekitar titik itu paling sulit bagi solver.

In [ ]:
n, trials = 30, 40
ratios = [3.0, 3.5, 4.0, 4.26, 4.5, 5.0, 5.5, 6.0]; psat, dec = [], []
for r in ratios:
    s, d = 0, 0
    for _ in range(trials):
        st = {"decisions": 0}; s += dpll(random_3sat(n, int(r * n)), st); d += st["decisions"]
    psat.append(s / trials); dec.append(d / trials)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(ratios, psat, "o-"); ax[0].set_title("P(satisfiable)"); ax[0].set_xlabel("m/n")
ax[1].plot(ratios, dec, "s-"); ax[1].set_title("rata-rata keputusan DPLL"); ax[1].set_xlabel("m/n"); plt.show()

## 3. Reduksi 3-SAT → Independent Set (⇒ NP-hard untuk Independent Set / Clique / Vertex Cover)
Untuk tiap klausa buat 3 simpul (satu per literal), hubungkan ketiganya (segitiga) dan hubungkan simpul literal komplementer. Formula *satisfiable* ⇔ ada independent set berukuran m.

In [ ]:
def reduce_to_is(clauses):
    nodes = [(ci, l) for ci, c in enumerate(clauses) for l in c]; edges = set()
    for a, b in itertools.combinations(range(len(nodes)), 2):
        (ca, la), (cb, lb) = nodes[a], nodes[b]
        if ca == cb or la == -lb: edges.add((a, b))
    return nodes, edges

def has_independent_set(nv, edges, k):
    adj = defaultdict(set)
    for a, b in edges: adj[a].add(b); adj[b].add(a)
    return any(all(b not in adj[a] for a, b in itertools.combinations(sub, 2)) for sub in itertools.combinations(range(nv), k))

agree = 0
for _ in range(60):
    n = 4; cl = random_3sat(n, 4); nodes, edges = reduce_to_is(cl)
    agree += brute_sat(cl, n) == has_independent_set(len(nodes), edges, len(cl))
print(f"reduksi benar pada {agree}/60 instans acak")

## 4. [Modern] Inferensi klausa Horn — forward chaining O(ukuran basis aturan)
Entailment proposisional umum itu co-NP-complete, tetapi untuk klausa Horn (aturan "jika A dan B maka C") cukup waktu **linear** — alasan sistem pakar berbasis aturan praktis.

In [ ]:
def forward_chain(rules, facts):
    count = [len(p) for p, _ in rules]; watch = defaultdict(list)
    for i, (prem, _) in enumerate(rules):
        for p in prem: watch[p].append(i)
    known, q = set(), deque(facts)
    while q:
        f = q.popleft()
        if f in known: continue
        known.add(f)
        for i in watch[f]:
            count[i] -= 1
            if count[i] == 0: q.append(rules[i][1])
    return known

rules = [({"demam", "batuk"}, "flu"), ({"flu", "sesak"}, "periksa_dokter"), ({"ruam", "demam"}, "campak"),
         ({"campak"}, "isolasi"), ({"flu"}, "istirahat")]
print(sorted(forward_chain(rules, {"demam", "batuk", "sesak"})))

def chain_rules(n): return [({f"p{i}"}, f"p{i+1}") for i in range(n)]
for n in [10**4, 10**5, 10**6]:
    t = time.perf_counter(); k = forward_chain(chain_rules(n), {"p0"}); print(f"{n:>8} aturan → {len(k)} fakta, {time.perf_counter()-t:.3f}s")

**Catatan:** algoritma **Rete** (Forgy, 1982) menghindari pencocokan ulang seluruh aturan terhadap seluruh fakta pada setiap siklus (pendekatan naif bisa berbiaya O(R·F^k) untuk aturan dengan k kondisi) dengan menyimpan hasil pencocokan parsial dalam jaringan. Silakan eksplorasi CLIPS atau `experta` untuk percobaan lanjutan.

## 5. Latihan / Tugas
1. Tambahkan **pure literal elimination** dan heuristik pemilihan variabel (literal paling sering muncul) pada DPLL; bandingkan jumlah keputusan.
2. Implementasikan **backward chaining** untuk basis aturan di atas dan bandingkan jumlah aturan yang dievaluasi.
3. Reduksi 3-SAT → Vertex Cover (komplemen independent set: VC = V \ IS) dan verifikasi.
4. **Tugas:** klasifikasikan (P / NP-complete / belum diketahui) lima persoalan: 2-SAT, 3-SAT, Shortest Path, Hamiltonian Cycle, Graph Isomorphism; lalu analisis kompleksitas mesin inferensi sistem pakar mini buatan Anda.

In [ ]:
def backward_chain(rules, facts, goal):
    pass  # TODO